El Flip-Flop tipo D (DFF) es un dispositivo de almacenamiento secuencial de 1 bit. Se usa como bloque fundamental para construir registros y memorias.


```
out(t) = in(t-1)
```
Para una entrada de un ciclo de reloj t, retorna el valor anterior guardado.




```
def DFF(input):
  a = 0;
  b = input;
  return a;
  a = b;

#Si dejara cambiar el valor de a despues de retornar el anterior
```



In [1]:
class DFF:
    def __init__(self):
        # DFF inicial 0
        self.q = 0  # Estado interno del Flip-Flop
    def clock_tick(self, d): #Simulacion ticks de un reloj
        prev_q = self.q  # Guarda el estado anterior
        self.q = d  # Almacena el nuevo valor para el próximo ciclo
        return prev_q  # Retorna el estado anterior



In [2]:
def Not (a):
  if (a == 1):
    return 0
  else:
    return 1

def Xor (a,b):
  nota = Not(a)
  notb = Not(b)
  aAndNotb = And(a,notb)
  notaAndb = And(nota,b)
  return Or(aAndNotb,notaAndb)

def And (a,b):
  if(a == 1 and b == 1):
    return 1
  else:
    return 0

def Nand (a,b):
  return Not(And(a,b))

def Or (a,b):
  if(a == 1 or b == 1):
    return 1
  else:
    return 0

def Mux (a,b,load):
  a_and_load = And(a,load)
  load_not = Not(load)
  b_and_notload = And(b,load_not)
  return Or(a_and_load,b_and_notload)

# Si load es 1 pasa a, si load es 0 pasa b


In [3]:
class Bit:
    def __init__(self):
        self.dff = DFF()  # Se inicializa DFF para guardar el bit

    def update(self, data, load):
        prev_value = self.dff.q  # Estado actual del DFF, siendo 0 "valor inicial"
        mux_out = Mux(data,prev_value,load) #load 0 mantiene valor anterior load 1 se guarda el valora actual
        return self.dff.clock_tick(mux_out) #Se guarda en DFF





```
def Register16(input,load):
  result = []
  for i in range(15, -1, -1):
       bit_updated = bit.update(input[i],load)
       result.insert(0,bit_updated)
  return result

#Pero no se puede porque cada bit debe mantener su estado anterior, cada bit debe ser una instancia diferente hpta vida toca hacer otra clase

print(Register16([1]*16, 1))
print(Register16([0]*16, 0))
print(Register16([0]*16, 1))
print(Register16([0,1]*8, 1))
print(Register16([0,1]*8, 0))

[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0]
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
[1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
```



In [4]:
class Register:
    def __init__(self):
        self.bits = [Bit() for _ in range(16)]  # 16 bits

    def update(self, data, load):  # Parametros data, load, data input, load puede ser 0 o 1, si es 1 carga todos los datos, si es 0 no guarda nada inicializa todo como 000.. 16 bits
      new_value = [self.bits[i].update(data[i], load) for i in range(16)]  # Primero actualiza los bits
      return new_value




RAM8

8 Registros de 16 bits

Entradas: in[16], load, address[8] -> Selecciona el registro el cual se va a escribir

Usamos un demultiplexor para enviar el load al registro.
Usamos multiplexor para seleccionar la salida.

In [46]:

# Demultiplexores corregidos
def DMux(input, sel):    #Segun el select dirige la entrada a la salida correspondiente, en este caso es un DMux 1 - 2
    return (And(input, Not(sel)), And(input, sel))  # Si el select es 0 return 0, si select es 1 return input.


def DMux4Way(input, sel): #Select posee dos bits 00 - 11
    if len(sel) < 2:
        raise ValueError("DMux4Way necesita al menos 2 bits de selección.")

    inp0, inp1 = DMux(input, sel[0])
    a, b = DMux(inp0, sel[1])
    c, d = DMux(inp1, sel[1])

    return a, b, c, d


def DMux8Way(input, sel):
    """DMux 1:8 - Divide la entrada en 8 salidas según los 3 bits de selección."""
    if len(sel) < 3:
        raise ValueError("DMux8Way necesita al menos 3 bits de selección.")

    inp0, inp1 = DMux(input, sel[0])  # Primera división en 2

    # Segunda división en 4 (usando DMux4Way)
    a, b, c, d = DMux4Way(inp0, [sel[1], sel[2]])
    e, f, g, h = DMux4Way(inp1, [sel[1], sel[2]])

    return a, b, c, d, e, f, g, h


def Mux16(a, b, sel):
    """ Multiplexor 2:1 de 16 bits """
    return a if sel == 0 else b


def Mux4Way16(a, b, c, d, sel):
    """ Multiplexor 4:1 de 16 bits """
    mux1 = Mux16(a, b, sel[1])
    mux2 = Mux16(c, d, sel[1])
    return Mux16(mux1, mux2, sel[0])  # sel[0] es el bit más significativo


def Mux8Way16(a, b, c, d, e, f, g, h, sel):
    """ Multiplexor 8:1 de 16 bits """
    mux1 = Mux4Way16(a, c, b, d, [sel[2], sel[1]])  # El bit más significativo va primero
    mux2 = Mux4Way16(e, g, f, h, [sel[2], sel[1]])
    return Mux16(mux1, mux2, sel[0])  # El bit más significativo de Mux8Way16 es sel[0]

In [47]:
class TestMultiplexers:
    @staticmethod
    def test_mux16():
        """ Prueba Mux16 con valores de 16 bits """
        assert Mux16([1]*16, [0]*16, 0) == [1]*16, "Mux16(0) failed"
        assert Mux16([1]*16, [0]*16, 1) == [0]*16, "Mux16(1) failed"
        print("✔ Mux16 tests passed.")

    @staticmethod
    def test_mux4way16():
        """ Prueba Mux4Way16 con selección de 2 bits """
        inputs = [[i]*16 for i in range(4)]
        for i in range(4):
            sel = [int(x) for x in f"{i:02b}"]  # Convierte i en binario
            assert Mux4Way16(*inputs, sel) == [i]*16, f"Mux4Way16({sel}) failed"
        print("✔ Mux4Way16 tests passed.")

    @staticmethod
    def test_mux8way16():
        """ Prueba Mux8Way16 con selección de 3 bits """
        inputs = [[i]*16 for i in range(8)]
        print(inputs)
        for i in range(8):
            sel = [int(x) for x in f"{i:03b}"]  # Convierte i en binario
            print(sel)
            result = Mux8Way16(*inputs, sel)
            print(result)
            assert Mux8Way16(*inputs, sel) == [i]*16, f"Mux8Way16({sel}) failed"
        print("✔ Mux8Way16 tests passed.")

    @staticmethod
    def test_dmux():
        """ Prueba DMux 1:2 """
        assert DMux(1, 0) == (1, 0), "DMux(1, 0) failed"
        assert DMux(1, 1) == (0, 1), "DMux(1, 1) failed"
        assert DMux(0, 0) == (0, 0), "DMux(0, 0) failed"
        assert DMux(0, 1) == (0, 0), "DMux(0, 1) failed"
        print("✔ DMux tests passed.")

    @staticmethod
    def test_dmux4way():
        """ Prueba DMux 1:4 con selección de 2 bits """
        for i in range(4):
            sel = [int(x) for x in f"{i:02b}"]
            expected = [1 if j == i else 0 for j in range(4)]
            assert DMux4Way(1, sel) == tuple(expected), f"DMux4Way(1, {sel}) failed"
        print("✔ DMux4Way tests passed.")

    @staticmethod
    def test_dmux8way():
        """ Prueba DMux 1:8 con selección de 3 bits """
        for i in range(8):
            sel = [int(x) for x in f"{i:03b}"]
            expected = [1 if j == i else 0 for j in range(8)]
            assert DMux8Way(1, sel) == tuple(expected), f"DMux8Way(1, {sel}) failed"
        print("✔ DMux8Way tests passed.")

    @staticmethod
    def run_tests():
        """ Ejecuta todas las pruebas """
        TestMultiplexers.test_mux16()
        TestMultiplexers.test_mux4way16()
        TestMultiplexers.test_mux8way16()
        TestMultiplexers.test_dmux()
        TestMultiplexers.test_dmux4way()
        TestMultiplexers.test_dmux8way()
        print("🎉 ¡Todas las pruebas pasaron con éxito!")

# Ejecutar las pruebas
TestMultiplexers.run_tests()

✔ Mux16 tests passed.
✔ Mux4Way16 tests passed.
[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7]]
[0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 1]
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[0, 1, 0]
[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]
[0, 1, 1]
[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]
[1, 0, 0]
[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]
[1, 0, 1]
[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5]
[1, 1, 0]
[6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6]
[1, 1, 1]
[7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7]
✔ Mux8Way16 tests passed.
✔ DMux tests passed.
✔ DMux4Way tests passed.
✔ DMux8

In [48]:
class RAM8:
    def __init__(self):
        self.registers = [Register() for _ in range(8)]  # 8 registros de 16 bits

    def update(self, data, load, address):
        if len(address) != 3:
            raise ValueError("La dirección en RAM8 debe tener 3 bits.")
        if len(data) != 16:
            raise ValueError("Los datos deben tener 16 bits.")

        load_signals = DMux8Way(load, address)
        register_outputs = [self.registers[i].update(data, load_signals[i]) for i in range(8)]
        return Mux8Way16(*register_outputs, address)

In [53]:


class RAM64:
    def __init__(self):
        self.rams = [RAM8() for _ in range(8)]

    def update(self, data, load, address):
        if len(address) != 6:
            raise ValueError("La dirección en RAM64 debe tener 6 bits.")

        load_signals = DMux8Way(load, address[3:])
        block_outputs = [self.rams[i].update(data, load_signals[i], address[:3]) for i in range(8)]
        return Mux8Way16(*block_outputs, address[3:])


class RAM512:
    def __init__(self):
        self.rams = [RAM64() for _ in range(8)]

    def update(self, data, load, address):
        if len(address) != 9:
            raise ValueError("La dirección en RAM512 debe tener 9 bits.")

        load_signals = DMux8Way(load, address[6:])
        block_outputs = [self.rams[i].update(data, load_signals[i], address[:6]) for i in range(8)]
        return Mux8Way16(*block_outputs, address[6:])


class RAM4K:
    def __init__(self):
        self.rams = [RAM512() for _ in range(8)]

    def update(self, data, load, address):
        if len(address) != 12:
            raise ValueError("La dirección en RAM4K debe tener 12 bits.")

        load_signals = DMux8Way(load, address[9:])
        block_outputs = [self.rams[i].update(data, load_signals[i], address[:9]) for i in range(8)]
        return Mux8Way16(*block_outputs, address[9:])


class RAM16K:  # RAM con 14 bits de dirección
    def __init__(self):
        self.rams = [RAM4K() for _ in range(4)]  # 4 RAM4K = 16K

    def update(self, data, load, address):
        if len(address) != 14:
            raise ValueError("La dirección en RAM16K debe tener 14 bits.")
        if len(data) != 16:
            raise ValueError("Los datos deben tener 16 bits.")

        load_signals = DMux4Way(load, address[12:14])  # Últimos 2 bits para seleccionar el bloque
        block_outputs = [self.rams[i].update(data, load_signals[i], address[:12]) for i in range(4)]  # Primeros 12 bits dentro del bloque
        return Mux4Way16(*block_outputs, address[12:14])  # Selecciona la salida final

class RAM64K:  # RAM con 16 bits de dirección
    def __init__(self):
        self.rams = [RAM16K() for _ in range(4)]  # 4 RAM16K = 64K

    def update(self, data, load, address):
        if len(address) != 16:
            raise ValueError("La dirección en RAM64K debe tener 16 bits.")
        if len(data) != 16:
            raise ValueError("Los datos deben tener 16 bits.")

        load_signals = DMux4Way(load, address[14:16])  # Últimos 2 bits para seleccionar el bloque
        block_outputs = [self.rams[i].update(data, load_signals[i], address[:14]) for i in range(4)]  # Primeros 14 bits dentro del bloque
        return Mux4Way16(*block_outputs, address[14:16])  # Selecciona la salida final

In [50]:
def HalfAdder(a, b):
    # Suma: XOR de a y b
    sum_bit = Xor(a, b)

    # Acarreo: AND de a y b
    carry_bit = And(a, b)

    return sum_bit, carry_bit

def FullAdder(a, b, carry_in):
    # Suma parcial (a + b)
    sum_temp, carry_temp1 = HalfAdder(a, b)

    # Suma final (sum_temp + carry_in)
    sum_bit, carry_temp2 = HalfAdder(sum_temp, carry_in)

    # Acarreo final (carry_temp1 OR carry_temp2)
    carry_out = Or(carry_temp1, carry_temp2)

    return sum_bit, carry_out

def Add16(a, b):
    result = []
    carry = 0

    # Suma bit por bit (de derecha a izquierda)
    for i in range(15, -1, -1):
        sum_bit, carry = FullAdder(a[i], b[i], carry)
        result.insert(0, sum_bit)  # Insertar al inicio para mantener el orden

    return result

In [55]:
class PC:
    def __init__(self):
        self.register = Register()
        self.current_value = [0]*16  # Track current value

    def update(self, data, inc, load, reset):
        if reset:
            self.current_value = [0]*16
            self.register.update(self.current_value, 1)
            return self.current_value

        if inc:
            self.current_value = Add16(self.current_value, [0]*15 + [1])
            self.register.update(self.current_value, 1)
            return self.current_value

        if load:
            self.current_value = data.copy()
            self.register.update(self.current_value, 1)
            return self.current_value

        # No operation - return current value
        return self.current_value

In [54]:
import random

class TestRAM:
    @staticmethod
    def test_ram8():
        print("Probando RAM8...")
        ram = RAM8()
        address = random.randint(0, 7)  # Dirección en el rango [0,7]
        address_list = [int(bit) for bit in f"{address:03b}"]  # Convertir a lista de bits
        data = [random.randint(0, 1) for _ in range(16)]  # Datos aleatorios de 16 bits

        print(f"Dirección: {address_list} ({address}), Datos escritos: {data}")
        ram.update(data, 1, address_list)  # Escribir

        result = ram.update([0]*16, 0, address_list)  # Leer
        print(f"Datos leídos: {result}")

        assert result == data, f"Error en RAM8. Esperado: {data}, Obtenido: {result}"
        print("RAM8 funciona correctamente.\n")

    @staticmethod
    def test_ram64():
        print("Probando RAM64...")
        ram = RAM64()
        address = random.randint(0, 63)  # Dirección en el rango [0,63]
        address_list = [int(bit) for bit in f"{address:06b}"]  # Convertir a lista de bits
        data = [random.randint(0, 1) for _ in range(16)]

        print(f"Dirección: {address_list} ({address}), Datos escritos: {data}")
        ram.update(data, 1, address_list)

        result = ram.update([0]*16, 0, address_list)
        print(f"Datos leídos: {result}")

        assert result == data, f"Error en RAM64. Esperado: {data}, Obtenido: {result}"
        print("RAM64 funciona correctamente.\n")

    @staticmethod
    def test_ram512():
        print("Probando RAM512...")
        ram = RAM512()
        address = random.randint(0, 511)  # Dirección en el rango [0,511]
        address_list = [int(bit) for bit in f"{address:09b}"]  # Convertir a lista de bits
        data = [random.randint(0, 1) for _ in range(16)]  # Datos aleatorios de 16 bits

        print(f"Dirección: {address_list}, Datos escritos: {data}")
        ram.update(data, 1, address_list)  # Escribir
        result = ram.update([0]*16, 0, address_list)  # Leer
        print(f"Resultado: {result}, Data: {data}")
        assert result == data, "Error en RAM512"
        print("RAM512 funciona correctamente.\n")

    @staticmethod
    def test_ram4k():
        print("Probando RAM4K...")
        ram = RAM4K()
        address = random.randint(0, 4095)  # Dirección en el rango [0,4095]
        address_list = [int(bit) for bit in f"{address:012b}"]  # Convertir a lista de bits
        data = [random.randint(0, 1) for _ in range(16)]

        print(f"Dirección: {address_list}, Datos escritos: {data}")
        ram.update(data, 1, address_list)
        result = ram.update([0]*16, 0, address_list)
        print(f"Resultado: {result}, Data: {data}")
        assert result == data, "Error en RAM4K"
        print("RAM4K funciona correctamente.\n")

    @staticmethod
    def test_ram16k():
        print("Probando RAM16K...")
        ram = RAM16K()
        address = random.randint(0, 16383)  # Dirección en el rango [0,16383]
        address_list = [int(bit) for bit in f"{address:014b}"]  # Convertir a lista de bits
        data = [random.randint(0, 1) for _ in range(16)]

        print(f"Dirección: {address_list}, Datos escritos: {data}")
        ram.update(data, 1, address_list)
        result = ram.update([0]*16, 0, address_list)
        print(f"Resultado: {result}, Data: {data}")
        assert result == data, "Error en RAM16K"
        print("RAM16K funciona correctamente.\n")

    @staticmethod
    def test_ram64k():
      print("Probando RAM64K...")
      ram = RAM64K()
      address = random.randint(0, 65535)  # Dirección en el rango [0, 65535]
      address_list = [int(bit) for bit in f"{address:016b}"]  # Convertir a lista de bits
      data = [random.randint(0, 1) for _ in range(16)]  # Datos de 16 bits aleatorios

      print(f"Dirección: {address_list}, Datos escritos: {data}")
      ram.update(data, 1, address_list)  # Escribir datos
      result = ram.update([0]*16, 0, address_list)  # Leer datos

      print(f"Resultado: {result}, Data: {data}")
      assert result == data, "Error en RAM64K"
      print("RAM64K funciona correctamente.\n")


# EJECUTAR PRUEBAS
if __name__ == "__main__":
    TestRAM.test_ram8()
    TestRAM.test_ram64()
    TestRAM.test_ram512()
    TestRAM.test_ram4k()
    TestRAM.test_ram16k()
    TestRAM.test_ram64k()

Probando RAM8...
Dirección: [1, 0, 0] (4), Datos escritos: [1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0]
Datos leídos: [1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0]
RAM8 funciona correctamente.

Probando RAM64...
Dirección: [0, 1, 1, 1, 0, 0] (28), Datos escritos: [1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0]
Datos leídos: [1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0]
RAM64 funciona correctamente.

Probando RAM512...
Dirección: [0, 0, 1, 1, 1, 1, 1, 0, 0], Datos escritos: [1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1]
Resultado: [1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1], Data: [1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1]
RAM512 funciona correctamente.

Probando RAM4K...
Dirección: [1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0], Datos escritos: [1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0]
Resultado: [1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0], Data: [1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0]
RAM4K funciona correctamente.

Probando RAM16K...
Dirección: [1, 0